# MODULE 9 — HANDS-ON PROJECT
# Customer & Sales Analytics System

---

> **Scenario:**  
> You've just joined **TechMart** as a Senior Data Analyst.  
> The Head of Sales hands you a messy Excel dump from their CRM system.  
> Your task: clean it, compute KPIs, extract business insights, and produce a
> **dashboard-ready dataset** for the BI team by end of week.

**Deliverables:**
1. Clean dataset (gold-layer)
2. KPI summary table
3. Business insights report
4. Dashboard-ready aggregated tables
5. Anomaly detection

---

In [20]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.width', 120)

print('Libraries loaded. Starting TechMart Analytics Project...')

Libraries loaded. Starting TechMart Analytics Project...


### STEP 1 — Generate the Raw Messy Dataset
*This simulates the dirty CRM export you would receive from a real system.*

In [21]:
np.random.seed(2024)
N = 2000  # 2,000 orders

# ── Raw CRM Export (as messy as real data gets) ───────────────────────────────
products_catalog = {
    'Laptop Pro'     : 1299, 'Laptop Basic': 699,
    'Smartphone X'   : 899,  'Smartphone Lite': 449,
    'Tablet Premium' : 799,  'Tablet Standard': 399,
    'Monitor 27"'    : 549,  'Monitor 24"': 349,
    'Mechanical Keyboard': 149, 'Wireless Mouse': 59,
    'USB Hub'        : 49,   'Webcam HD': 99
}

product_names    = list(products_catalog.keys())
product_prices   = list(products_catalog.values())
product_category = ['Computing','Computing','Mobile','Mobile','Computing',
                    'Computing','Computing','Computing','Peripheral',
                    'Peripheral','Peripheral','Peripheral']

# Customer base
n_customers = 300
first_names = ['Alice','Bob','Carol','David','Eve','Frank','Grace','Henry',
               'Ivy','Jack','Karen','Leo','Mia','Noah','Olivia','Paul',
               'Quinn','Rachel','Steve','Tina','Uma','Victor','Wendy','Xander']
last_names  = ['Smith','Johnson','Williams','Brown','Jones','Miller','Davis',
               'Wilson','Moore','Taylor','Anderson','Thomas','Jackson',
               'White','Harris','Martin','Thompson','Garcia','Martinez','Lee']

rng = np.random.default_rng(2024)
customer_pool = pd.DataFrame({
    'customer_id'  : range(1001, 1001 + n_customers),
    'customer_name': [f"{rng.choice(first_names)} {rng.choice(last_names)}" for _ in range(n_customers)],
    'email'        : [f"user{i}@{'gmail' if i%3==0 else 'yahoo' if i%3==1 else 'corp'}.com"
                      for i in range(n_customers)],
    'segment'      : rng.choice(['Enterprise', 'SMB', 'Consumer', 'Education'],
                                 n_customers, p=[0.15, 0.30, 0.40, 0.15]),
    'region'       : rng.choice(['North America', 'Europe', 'Asia Pacific', 'LATAM'],
                                 n_customers, p=[0.40, 0.30, 0.20, 0.10]),
    'account_manager': rng.choice(['Alice Chen','Bob Kumar','Carol White',
                                    'David Park','Eve Russo'], n_customers)
})

# Generate orders
order_customer_ids = rng.choice(customer_pool['customer_id'], N)
order_product_idx  = rng.choice(len(product_names), N)

raw_data = pd.DataFrame({
    'Order ID'        : [f'ORD-{100000+i}' for i in range(N)],
    'Customer ID'     : order_customer_ids,
    'Customer Name'   : [customer_pool.loc[customer_pool['customer_id']==cid,'customer_name'].values[0]
                         for cid in order_customer_ids],
    'Product'         : [product_names[i] for i in order_product_idx],
    'Category'        : [product_category[i] for i in order_product_idx],
    'Quantity'        : rng.integers(1, 8, N),
    'Unit Price'      : [product_prices[i] for i in order_product_idx],
    'Discount%'       : rng.choice([0, 5, 10, 15, 20, 25], N, p=[0.3,0.25,0.2,0.1,0.1,0.05]),
    'Order Date'      : [str(pd.Timestamp('2023-01-01') + pd.Timedelta(days=int(d)))
                         for d in rng.integers(0, 730, N)],
    'Payment Method'  : rng.choice(['Credit Card','Bank Transfer','PayPal','Invoice'],
                                    N, p=[0.4,0.3,0.2,0.1]),
    'Ship Country'    : rng.choice(['US','USA','United States','UK','United Kingdom',
                                     'Germany','DE','France','Canada','CA','Japan','JP'],
                                    N, p=[0.20,0.10,0.10,0.05,0.05,0.10,0.05,
                                          0.05,0.05,0.05,0.10,0.10]),
    'Status'          : rng.choice(['Completed','Pending','Shipped','Cancelled','Returned'],
                                    N, p=[0.65, 0.10, 0.15, 0.05, 0.05]),
    'Notes'           : rng.choice(['', 'Urgent', 'Gift wrap', 'Corporate account',
                                     None, 'Reseller'], N)
})

# ── Inject realistic data quality issues ──────────────────────────────────────
# 1. Missing values
null_idx = rng.choice(N, 120, replace=False)
raw_data.loc[null_idx[:40], 'Unit Price']    = np.nan
raw_data.loc[null_idx[40:80], 'Order Date']  = np.nan
raw_data.loc[null_idx[80:], 'Payment Method'] = np.nan

# 2. Duplicate orders
dup_rows = raw_data.sample(15, random_state=42)
raw_data = pd.concat([raw_data, dup_rows], ignore_index=True)

# 3. Negative/zero quantities
raw_data.loc[rng.choice(N, 20, replace=False), 'Quantity'] = rng.choice([-1, 0], 20)

# 4. Price errors (outliers)
raw_data.loc[rng.choice(N, 5, replace=False), 'Unit Price'] = rng.choice([0.01, 99999], 5)

# 5. Mixed date formats
mixed_date_idx = rng.choice(N, 50, replace=False)
for idx in mixed_date_idx:
    dt = pd.Timestamp('2023-01-01') + pd.Timedelta(days=int(rng.integers(0, 730)))
    raw_data.loc[idx, 'Order Date'] = dt.strftime('%d/%m/%Y') if idx % 2 == 0 else dt.strftime('%B %d, %Y')

print(f'Raw dataset generated: {raw_data.shape[0]} rows, {raw_data.shape[1]} columns')
print(f'(Including {len(dup_rows)} duplicate rows and various data quality issues)')
raw_data.head(8)

Raw dataset generated: 2015 rows, 13 columns
(Including 15 duplicate rows and various data quality issues)


,Order ID,Customer ID,Customer Name,Product,Category,Quantity,Unit Price,Discount%,Order Date,Payment Method,Ship Country,Status,Notes
0,ORD-100000,1087,Grace Williams,Webcam HD,Peripheral,3,99.00,10,2023-01-08 00:00:00,Bank Transfer,Germany,Pending,NaN
1,ORD-100001,1193,Xander Martin,Webcam HD,Peripheral,1,99.00,0,2023-01-29 00:00:00,PayPal,US,Completed,Reseller
2,ORD-100002,1143,Mia Brown,"Monitor 24""",Computing,3,349.00,0,2023-05-12 00:00:00,Invoice,United States,Completed,Reseller
3,ORD-100003,1198,Carol Lee,"Monitor 24""",Computing,7,349.00,25,2024-12-18 00:00:00,Credit Card,Germany,Completed,Corporate account
4,ORD-100004,1159,Rachel Lee,Mechanical Keyboard,Peripheral,6,149.00,0,2023-06-30 00:00:00,Bank Transfer,Germany,Pending,
5,ORD-100005,1044,Tina Thomas,Mechanical Keyboard,Peripheral,5,149.00,0,2024-11-25 00:00:00,PayPal,CA,Cancelled,Reseller
6,ORD-100006,1201,Karen White,Webcam HD,Peripheral,1,99.00,10,2024-07-28 00:00:00,Bank Transfer,JP,Completed,
7,ORD-100007,1077,Alice Harris,Laptop Basic,Computing,1,699.00,5,2024-04-03 00:00:00,Bank Transfer,United States,Completed,


### STEP 2 — Data Quality Audit (Before Cleaning)

In [22]:
def full_audit(df):
    """Comprehensive data quality audit — run before any cleaning."""
    print('='*60)
    print('DATA QUALITY AUDIT REPORT')
    print('='*60)
    print(f'Rows    : {df.shape[0]:,}')
    print(f'Columns : {df.shape[1]}')
    print(f'Memory  : {df.memory_usage(deep=True).sum()/1024**2:.2f} MB')
    print(f'Full duplicates : {df.duplicated().sum()}')
    print()

    audit = pd.DataFrame({
        'dtype'       : df.dtypes,
        'non_null'    : df.notna().sum(),
        'null_count'  : df.isnull().sum(),
        'null_pct'    : (df.isnull().mean() * 100).round(1),
        'unique'      : df.nunique(),
        'top_value'   : [df[c].mode()[0] if df[c].notna().any() else 'N/A' for c in df.columns]
    })
    return audit

full_audit(raw_data)

DATA QUALITY AUDIT REPORT
Rows    : 2,015
Columns : 13
Memory  : 0.38 MB
Full duplicates : 14



,dtype,non_null,null_count,null_pct,unique,top_value
Order ID,str,2015,0,0.00,2000,ORD-100056
Customer ID,int64,2015,0,0.00,299,1046
Customer Name,str,2015,0,0.00,229,Victor Jones
Product,str,2015,0,0.00,12,Laptop Pro
Category,str,2015,0,0.00,3,Computing
Quantity,int64,2015,0,0.00,9,4
Unit Price,float64,1975,40,2.00,14,"1,299.00"
Discount%,int64,2015,0,0.00,6,0
Order Date,str,1974,41,2.00,726,2023-10-24 00:00:00
Payment Method,str,1975,40,2.00,4,Credit Card


### STEP 3 — Data Cleaning Pipeline

In [23]:
def clean_techmart_data(raw):
    """
    Production-grade cleaning pipeline for TechMart CRM export.
    Returns: clean DataFrame ready for analysis
    """
    df = raw.copy()
    issues_log = []

    # ── Step 3.1: Standardize column names ────────────────────────────────────
    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace(' ', '_', regex=False)
                  .str.replace('%', '_pct', regex=False)
                  .str.replace('"', '', regex=False))
    issues_log.append('✓ Column names standardized')

    # ── Step 3.2: Remove full duplicates ──────────────────────────────────────
    n_before = len(df)
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    issues_log.append(f'✓ Removed {n_before - len(df)} duplicate rows')

    # ── Step 3.3: Fix Order Date ───────────────────────────────────────────────
    df['order_date'] = pd.to_datetime(df['order_date'],
                                      dayfirst=False,
                                      errors='coerce',
                                    #   infer_datetime_format=True
                                      )
    null_dates = df['order_date'].isnull().sum()
    # Impute with median date
    median_dt = df['order_date'].dropna().sort_values().iloc[len(df['order_date'].dropna())//2]
    df['order_date'].fillna(median_dt, inplace=True)
    issues_log.append(f'✓ Fixed {null_dates} unparseable order dates (imputed with median)')

    # ── Step 3.4: Fix Unit Price ───────────────────────────────────────────────
    # Fill missing prices with product median
    null_prices = df['unit_price'].isnull().sum()
    df['unit_price'] = df.groupby('product')['unit_price'].transform(
        lambda x: x.fillna(x.median())
    )
    # Remove price outliers (< $1 or > $50,000)
    price_outliers = ((df['unit_price'] < 1) | (df['unit_price'] > 50_000)).sum()
    df.loc[(df['unit_price'] < 1) | (df['unit_price'] > 50_000), 'unit_price'] = np.nan
    df['unit_price'] = df.groupby('product')['unit_price'].transform(
        lambda x: x.fillna(x.median())
    )
    issues_log.append(f'✓ Fixed {null_prices} missing prices, removed {price_outliers} price outliers')

    # ── Step 3.5: Fix invalid quantities ──────────────────────────────────────
    bad_qty = (df['quantity'] <= 0).sum()
    df = df[df['quantity'] > 0].copy()
    issues_log.append(f'✓ Removed {bad_qty} rows with invalid quantity (<= 0)')

    # ── Step 3.6: Standardize country names ───────────────────────────────────
    country_map = {
        'US': 'United States', 'USA': 'United States',
        'UK': 'United Kingdom',
        'DE': 'Germany',
        'CA': 'Canada',
        'JP': 'Japan'
    }
    df['ship_country'] = (df['ship_country']
                          .map(country_map)
                          .fillna(df['ship_country']))
    issues_log.append('✓ Country names normalized')

    # ── Step 3.7: Fill remaining nulls ────────────────────────────────────────
    df['payment_method'].fillna('Unknown', inplace=True)
    df['notes'].fillna('', inplace=True)
    issues_log.append('✓ Remaining nulls filled with sentinels')

    # ── Step 3.8: Compute revenue ─────────────────────────────────────────────
    df['gross_revenue'] = (df['quantity'] * df['unit_price']).round(2)
    df['net_revenue']   = (df['gross_revenue'] * (1 - df['discount_pct'] / 100)).round(2)
    df['discount_amount'] = (df['gross_revenue'] - df['net_revenue']).round(2)
    issues_log.append('✓ Revenue columns computed')

    # ── Step 3.9: Date features ───────────────────────────────────────────────
            
    df['order_year']    = df['order_date'].dt.year
    df['order_month']   = df['order_date'].dt.month
    df['order_quarter'] = df['order_date'].dt.quarter
    df['order_yearmon'] = df['order_date'].dt.to_period('M').astype(str)
    df['is_weekend']    = df['order_date'].dt.dayofweek >= 5

    # ── Step 3.10: Optimize dtypes ────────────────────────────────────────────
    cat_cols = ['product', 'category', 'ship_country', 'status',
                'payment_method', 'order_yearmon']
    for c in cat_cols:
        df[c] = df[c].astype('category')

    display(df)
    # for col in ['order_year', 'order_month', 'order_quarter']:
    #     if col in df.columns:
    #         df.fillna(0, inplace=True)
    
    # try:
    #     df['order_year'] = df['order_year'].str.replace({',': ''}).astype('int16')        
    # except:
    #     df['order_year']    = df['order_year'].astype('int8')
    df['order_year'] = df['order_year'].astype(str).str.replace({',': '', '.0': ''}).fillna('2025').astype('int16')  
    df['order_month']   = df['order_month'].astype(str).str.replace({'.0': ''}).fillna('0').astype('int8')
    df['order_quarter'] = df['order_quarter'].astype(str).str.replace({'.0': ''}).fillna('0').astype('int8')

    print('\n'.join(issues_log))
    return df

# Run the cleaning pipeline
clean = clean_techmart_data(raw_data)
print(f'\nClean dataset: {clean.shape[0]:,} rows × {clean.shape[1]} columns')

,order_id,customer_id,customer_name,product,category,quantity,unit_price,discount_pct,order_date,payment_method,ship_country,status,notes,gross_revenue,net_revenue,discount_amount,order_year,order_month,order_quarter,order_yearmon,is_weekend
0,ORD-100000,1087,Grace Williams,Webcam HD,Peripheral,3,99.00,10,2023-01-08,Bank Transfer,Germany,Pending,NaN,297.00,267.30,29.70,"2,023.00",1.00,1.00,2023-01,True
1,ORD-100001,1193,Xander Martin,Webcam HD,Peripheral,1,99.00,0,2023-01-29,PayPal,United States,Completed,Reseller,99.00,99.00,0.00,"2,023.00",1.00,1.00,2023-01,True
2,ORD-100002,1143,Mia Brown,"Monitor 24""",Computing,3,349.00,0,2023-05-12,Invoice,United States,Completed,Reseller,"1,047.00","1,047.00",0.00,"2,023.00",5.00,2.00,2023-05,False
3,ORD-100003,1198,Carol Lee,"Monitor 24""",Computing,7,349.00,25,2024-12-18,Credit Card,Germany,Completed,Corporate account,"2,443.00","1,832.25",610.75,"2,024.00",12.00,4.00,2024-12,False
4,ORD-100004,1159,Rachel Lee,Mechanical Keyboard,Peripheral,6,149.00,0,2023-06-30,Bank Transfer,Germany,Pending,,894.00,894.00,0.00,"2,023.00",6.00,2.00,2023-06,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1996,ORD-101996,1020,Ivy Davis,Laptop Basic,Computing,4,699.00,10,2023-10-04,PayPal,United States,Completed,Urgent,"2,796.00","2,516.40",279.60,"2,023.00",10.00,4.00,2023-10,False
1997,ORD-101997,1203,Alice Jackson,Tablet Premium,Computing,5,799.00,5,2024-05-27,Bank Transfer,France,Completed,Corporate account,"3,995.00","3,795.25",199.75,"2,024.00",5.00,2.00,2024-05,False
1998,ORD-101998,1025,Tina Jones,Webcam HD,Peripheral,4,99.00,0,2024-10-14,Bank Transfer,United Kingdom,Completed,Reseller,396.00,396.00,0.00,"2,024.00",10.00,4.00,2024-10,False
1999,ORD-101999,1105,Mia Jackson,Laptop Basic,Computing,7,699.00,10,2024-06-09,Bank Transfer,Japan,Completed,,"4,893.00","4,403.70",489.30,"2,024.00",6.00,2.00,2024-06,True


✓ Column names standardized
✓ Removed 14 duplicate rows
✓ Fixed 90 unparseable order dates (imputed with median)
✓ Fixed 40 missing prices, removed 5 price outliers
✓ Removed 20 rows with invalid quantity (<= 0)
✓ Country names normalized
✓ Remaining nulls filled with sentinels
✓ Revenue columns computed

Clean dataset: 1,981 rows × 21 columns


In [24]:
# Verify cleaning results
print('=== POST-CLEANING VALIDATION ===')
print(f'Nulls remaining: {clean.isnull().sum().sum()}')
print(f'Duplicate rows : {clean.duplicated().sum()}')
print(f'Invalid qty    : {(clean["quantity"] <= 0).sum()}')
print(f'Date range     : {clean["order_date"].min().date()} → {clean["order_date"].max().date()}')
print(f'Revenue range  : ${clean["net_revenue"].min():,.2f} → ${clean["net_revenue"].max():,.2f}')
print()
clean[['order_id', 'product', 'quantity', 'unit_price', 'discount_pct',
       'gross_revenue', 'net_revenue', 'order_date', 'status']].head(8)

=== POST-CLEANING VALIDATION ===
Nulls remaining: 532
Duplicate rows : 0
Invalid qty    : 0
Date range     : 2023-01-01 → 2024-12-30
Revenue range  : $36.75 → $9,093.00



,order_id,product,quantity,unit_price,discount_pct,gross_revenue,net_revenue,order_date,status
0,ORD-100000,Webcam HD,3,99.00,10,297.00,267.30,2023-01-08,Pending
1,ORD-100001,Webcam HD,1,99.00,0,99.00,99.00,2023-01-29,Completed
2,ORD-100002,"Monitor 24""",3,349.00,0,"1,047.00","1,047.00",2023-05-12,Completed
3,ORD-100003,"Monitor 24""",7,349.00,25,"2,443.00","1,832.25",2024-12-18,Completed
4,ORD-100004,Mechanical Keyboard,6,149.00,0,894.00,894.00,2023-06-30,Pending
5,ORD-100005,Mechanical Keyboard,5,149.00,0,745.00,745.00,2024-11-25,Cancelled
6,ORD-100006,Webcam HD,1,99.00,10,99.00,89.10,2024-07-28,Completed
7,ORD-100007,Laptop Basic,1,699.00,5,699.00,664.05,2024-04-03,Completed


### STEP 4 — KPI Calculation

In [25]:
# ── Filter to completed/shipped orders only for revenue KPIs ──────────────────
active = clean[clean['status'].isin(['Completed', 'Shipped'])].copy()

# ── Core Business KPIs ────────────────────────────────────────────────────────
kpis = {
    '01. Total Orders'          : len(active),
    '02. Unique Customers'      : active['customer_id'].nunique(),
    '03. Gross Revenue ($)'     : active['gross_revenue'].sum(),
    '04. Net Revenue ($)'       : active['net_revenue'].sum(),
    '05. Total Discount ($)'    : active['discount_amount'].sum(),
    '06. Discount Rate (%)'     : (active['discount_amount'].sum() / active['gross_revenue'].sum() * 100),
    '07. Avg Order Value ($)'   : active['net_revenue'].mean(),
    '08. Median Order Value ($)': active['net_revenue'].median(),
    '09. Orders per Customer'   : len(active) / active['customer_id'].nunique(),
    '10. Revenue per Customer'  : active['net_revenue'].sum() / active['customer_id'].nunique(),
    '11. Units Sold'            : active['quantity'].sum(),
    '12. Avg Units per Order'   : active['quantity'].mean(),
    '13. Cancellation Rate (%)'  : (clean['status'] == 'Cancelled').mean() * 100,
    '14. Return Rate (%)'        : (clean['status'] == 'Returned').mean() * 100,
    '15. Weekend Order Share (%)': active['is_weekend'].mean() * 100
}

kpi_df = pd.Series(kpis).to_frame('value')
kpi_df['value'] = kpi_df['value'].round(2)
print('='*50)
print('TECHMART — EXECUTIVE KPI DASHBOARD')
print('='*50)
for k, v in kpis.items():
    if '$' in k:
        print(f'{k:<35}: ${v:>12,.0f}')
    elif '%' in k:
        print(f'{k:<35}: {v:>11.1f}%')
    else:
        print(f'{k:<35}: {v:>12,.1f}')

TECHMART — EXECUTIVE KPI DASHBOARD
01. Total Orders                   :      1,578.0
02. Unique Customers               :        298.0
03. Gross Revenue ($)              : $   3,015,091
04. Net Revenue ($)                : $   2,783,742
05. Total Discount ($)             : $     231,349
06. Discount Rate (%)              :         7.7%
07. Avg Order Value ($)            : $       1,764
08. Median Order Value ($)         : $       1,223
09. Orders per Customer            :          5.3
10. Revenue per Customer           :      9,341.4
11. Units Sold                     :      6,209.0
12. Avg Units per Order            :          3.9
13. Cancellation Rate (%)          :         5.0%
14. Return Rate (%)                :         4.8%
15. Weekend Order Share (%)        :        27.0%


### STEP 5 — Business Insights Extraction

In [26]:
# ── Insight 1: Product Performance Matrix ─────────────────────────────────────
product_perf = active.groupby('product', observed=True).agg(
    orders       = ('order_id', 'count'),
    units_sold   = ('quantity', 'sum'),
    gross_rev    = ('gross_revenue', 'sum'),
    net_rev      = ('net_revenue', 'sum'),
    discount_amt = ('discount_amount', 'sum'),
    avg_discount = ('discount_pct', 'mean')
).round(2)

product_perf['revenue_share_%'] = (product_perf['net_rev'] / product_perf['net_rev'].sum() * 100).round(1)
product_perf['avg_order_val']   = (product_perf['net_rev'] / product_perf['orders']).round(0)
product_perf = product_perf.sort_values('net_rev', ascending=False)

print('=== PRODUCT PERFORMANCE MATRIX ===')
print(product_perf[['orders','units_sold','net_rev','revenue_share_%',
                     'avg_order_val','avg_discount']].to_string())

=== PRODUCT PERFORMANCE MATRIX ===
                     orders  units_sold    net_rev  revenue_share_%  avg_order_val  avg_discount
product                                                                                         
Laptop Pro              144         559 672,817.05            24.20       4,672.00          7.71
Laptop Basic            141         589 379,277.40            13.60       2,690.00          8.26
Tablet Premium          127         499 364,064.35            13.10       2,867.00          8.66
Smartphone X            117         420 347,058.95            12.50       2,966.00          7.78
Monitor 27"             135         533 269,394.30             9.70       1,996.00          7.78
Tablet Standard         143         566 212,048.55             7.60       1,483.00          6.85
Smartphone Lite         117         479 200,545.85             7.20       1,714.00          7.48
Monitor 24"             134         526 169,055.60             6.10       1,262.00          

In [27]:
# ── Insight 2: Monthly Revenue Trend ──────────────────────────────────────────
monthly_trend = active.groupby('order_yearmon', observed=True).agg(
    orders   = ('order_id', 'count'),
    net_rev  = ('net_revenue', 'sum'),
    customers = ('customer_id', 'nunique')
).reset_index()

monthly_trend['rev_mom_pct'] = monthly_trend['net_rev'].pct_change() * 100
monthly_trend['rev_cumsum']  = monthly_trend['net_rev'].cumsum()
monthly_trend['arpu']        = (monthly_trend['net_rev'] / monthly_trend['customers']).round(0)

print('=== MONTHLY REVENUE TREND ===')
print(monthly_trend.set_index('order_yearmon').round(1).to_string())

=== MONTHLY REVENUE TREND ===
               orders    net_rev  customers  rev_mom_pct   rev_cumsum     arpu
order_yearmon                                                                 
2023-01            66 131,985.70         59          NaN   131,985.70 2,237.00
2023-02            61 105,521.80         52       -20.10   237,507.50 2,029.00
2023-03            68 132,792.10         63        25.80   370,299.60 2,108.00
2023-04            56 104,787.10         52       -21.10   475,086.60 2,015.00
2023-05            61  99,243.50         57        -5.30   574,330.20 1,741.00
2023-06            58 100,353.40         54         1.10   674,683.50 1,858.00
2023-07            65 130,683.40         60        30.20   805,366.80 2,178.00
2023-08            59  96,125.00         54       -26.40   901,491.80 1,780.00
2023-09            65  86,566.80         59        -9.90   988,058.60 1,467.00
2023-10            78 128,426.60         70        48.40 1,116,485.20 1,835.00
2023-11            62 

In [28]:
# ── Insight 3: Customer Segmentation (RFM Analysis) ──────────────────────────
# RFM = Recency, Frequency, Monetary — the gold standard of customer segmentation
snapshot_date = active['order_date'].max() + pd.Timedelta(days=1)

rfm = active.groupby('customer_id').agg(
    recency   = ('order_date',   lambda x: (snapshot_date - x.max()).days),
    frequency = ('order_id',     'count'),
    monetary  = ('net_revenue',  'sum')
).round(2)

# Score each dimension 1-5
rfm['R'] = pd.qcut(rfm['recency'],   q=5, labels=[5,4,3,2,1]).astype(int)  # lower=better
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'].rank(method='first'),  q=5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_Score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm['RFM_Total'] = rfm[['R','F','M']].sum(axis=1)

# Segment customers
def rfm_segment(row):
    if row['RFM_Total'] >= 13: return 'Champions'
    elif row['RFM_Total'] >= 10: return 'Loyal Customers'
    elif row['RFM_Total'] >= 7:  return 'Potential Loyalists'
    elif row['RFM_Total'] >= 5:  return 'At Risk'
    else: return 'Lost'

rfm['segment'] = rfm.apply(rfm_segment, axis=1)

seg_summary = rfm.groupby('segment').agg(
    customer_count = ('recency', 'count'),
    avg_recency    = ('recency', 'mean'),
    avg_frequency  = ('frequency', 'mean'),
    avg_monetary   = ('monetary', 'mean'),
    total_revenue  = ('monetary', 'sum')
).round(1).sort_values('total_revenue', ascending=False)

print('=== RFM CUSTOMER SEGMENTATION ===')
print(seg_summary.to_string())

=== RFM CUSTOMER SEGMENTATION ===
                     customer_count  avg_recency  avg_frequency  avg_monetary  total_revenue
segment                                                                                     
Loyal Customers                  80        93.10           6.40     12,396.70     991,736.00
Champions                        53        45.20           8.20     15,683.80     831,241.60
Potential Loyalists              91       129.90           4.60      7,428.30     675,978.80
At Risk                          43       184.70           3.40      4,686.60     201,522.30
Lost                             31       325.90           2.10      2,685.90      83,263.20


In [29]:
# ── Insight 4: Top 10% vs Bottom 10% Customers (Pareto Analysis) ─────────────
customer_revenue = active.groupby('customer_id')['net_revenue'].sum().sort_values(ascending=False)
total_rev = customer_revenue.sum()

top_10_pct_n    = max(1, int(len(customer_revenue) * 0.10))
top10_rev       = customer_revenue.head(top_10_pct_n).sum()
top10_share     = top10_rev / total_rev * 100

print('=== PARETO ANALYSIS ===')
print(f'Total customers : {len(customer_revenue):,}')
print(f'Top 10% = {top_10_pct_n} customers → ${top10_rev:,.0f} revenue ({top10_share:.1f}% of total)')
print(f'\nTop 10 customers by revenue:')
print(customer_revenue.head(10).apply(lambda x: f'${x:,.0f}').to_string())

=== PARETO ANALYSIS ===
Total customers : 298
Top 10% = 29 customers → $621,302 revenue (22.3% of total)

Top 10 customers by revenue:
customer_id
1131    $30,738
1012    $30,186
1103    $28,597
1082    $28,424
1250    $26,844
1272    $24,982
1243    $22,748
1187    $22,402
1122    $22,375
1041    $21,780


In [30]:
# ── Insight 5: Payment Method & Discount Analysis ─────────────────────────────
payment_analysis = active.groupby('payment_method', observed=True).agg(
    orders      = ('order_id', 'count'),
    net_rev     = ('net_revenue', 'sum'),
    avg_disc    = ('discount_pct', 'mean'),
    avg_qty     = ('quantity', 'mean')
).round(2)
payment_analysis['share_%'] = (payment_analysis['net_rev'] / payment_analysis['net_rev'].sum() * 100).round(1)
payment_analysis = payment_analysis.sort_values('net_rev', ascending=False)
print('=== PAYMENT METHOD ANALYSIS ===')
print(payment_analysis.to_string())

=== PAYMENT METHOD ANALYSIS ===
                orders      net_rev  avg_disc  avg_qty  share_%
payment_method                                                 
Credit Card        603 1,018,124.75      8.04     3.81    37.50
Bank Transfer      466   787,905.30      8.00     3.91    29.00
PayPal             315   616,306.65      7.60     4.10    22.70
Invoice            160   293,633.75      6.78     4.13    10.80


In [31]:
# ── Insight 6: Country Performance ────────────────────────────────────────────
country_perf = active.groupby('ship_country', observed=True).agg(
    orders    = ('order_id', 'count'),
    customers = ('customer_id', 'nunique'),
    net_rev   = ('net_revenue', 'sum'),
    avg_order = ('net_revenue', 'mean')
).round(2)
country_perf['rev_share_%'] = (country_perf['net_rev'] / country_perf['net_rev'].sum() * 100).round(1)
country_perf = country_perf.sort_values('net_rev', ascending=False)
print('=== COUNTRY REVENUE BREAKDOWN ===')
print(country_perf.to_string())

=== COUNTRY REVENUE BREAKDOWN ===
                orders  customers      net_rev  avg_order  rev_share_%
ship_country                                                          
United States      601        252 1,043,646.15   1,736.52        37.50
Japan              334        199   622,946.35   1,865.11        22.40
Germany            248        173   419,545.15   1,691.71        15.10
United Kingdom     164        120   302,277.95   1,843.16        10.90
Canada             151        118   270,893.00   1,793.99         9.70
France              80         69   124,433.35   1,555.42         4.50


In [32]:
# ── Insight 7: Anomaly Detection ──────────────────────────────────────────────
print('=== ANOMALY DETECTION ===')

# 1. Revenue anomalies using Z-score
active2 = active.copy()
mean_rev = active2['net_revenue'].mean()
std_rev  = active2['net_revenue'].std()
active2['rev_zscore'] = ((active2['net_revenue'] - mean_rev) / std_rev).round(2)
anomalies = active2[active2['rev_zscore'].abs() > 3]
print(f'Revenue outlier orders (|Z| > 3): {len(anomalies)}')
if len(anomalies) > 0:
    print(anomalies[['order_id', 'product', 'quantity', 'net_revenue', 'rev_zscore']].to_string(index=False))

# 2. Single customer ordering huge volumes
cust_orders = active.groupby('customer_id')['order_id'].count()
Q3 = cust_orders.quantile(0.75)
IQR = Q3 - cust_orders.quantile(0.25)
whale_customers = cust_orders[cust_orders > Q3 + 3 * IQR]
print(f'\nWhale customers (extreme order count): {len(whale_customers)}')
print(whale_customers.head())

# 3. Days with zero orders
date_range = pd.date_range(active['order_date'].min(), active['order_date'].max(), freq='D')
active_dates = active['order_date'].dt.normalize()
missing_days = date_range[~date_range.isin(active_dates)]
print(f'\nDays with zero orders: {len(missing_days)}')

=== ANOMALY DETECTION ===
Revenue outlier orders (|Z| > 3): 29
  order_id    product  quantity  net_revenue  rev_zscore
ORD-100016 Laptop Pro         7     8,183.70        3.67
ORD-100048 Laptop Pro         7     7,274.40        3.15
ORD-100119 Laptop Pro         6     7,794.00        3.44
ORD-100179 Laptop Pro         7     8,183.70        3.67
ORD-100234 Laptop Pro         6     7,794.00        3.44
ORD-100235 Laptop Pro         7     8,183.70        3.67
ORD-100255 Laptop Pro         7     9,093.00        4.19
ORD-100282 Laptop Pro         6     7,794.00        3.44
ORD-100314 Laptop Pro         7     9,093.00        4.19
ORD-100458 Laptop Pro         7     8,183.70        3.67
ORD-100527 Laptop Pro         7     7,729.05        3.41
ORD-100600 Laptop Pro         6     7,794.00        3.44
ORD-100704 Laptop Pro         7     8,638.35        3.93
ORD-100765 Laptop Pro         7     7,729.05        3.41
ORD-100768 Laptop Pro         6     7,794.00        3.44
ORD-100812 Laptop Pro    

### STEP 6 — Dashboard-Ready Dataset Creation

In [33]:
# ── 6A: Monthly KPI Table (for line charts) ───────────────────────────────────
dash_monthly = active.groupby('order_yearmon', observed=True).agg(
    orders          = ('order_id', 'count'),
    unique_customers= ('customer_id', 'nunique'),
    gross_revenue   = ('gross_revenue', 'sum'),
    net_revenue     = ('net_revenue', 'sum'),
    discount_total  = ('discount_amount', 'sum'),
    units_sold      = ('quantity', 'sum'),
    avg_order_value = ('net_revenue', 'mean')
).round(2).reset_index()

dash_monthly['discount_rate']  = (dash_monthly['discount_total'] / dash_monthly['gross_revenue'] * 100).round(2)
dash_monthly['rev_mom_pct']    = dash_monthly['net_revenue'].pct_change().mul(100).round(1)
dash_monthly['orders_mom_pct'] = dash_monthly['orders'].pct_change().mul(100).round(1)

print('Monthly Dashboard Table (first 6 months):')
print(dash_monthly.head(6).to_string(index=False))

Monthly Dashboard Table (first 6 months):
order_yearmon  orders  unique_customers  gross_revenue  net_revenue  discount_total  units_sold  avg_order_value  discount_rate  rev_mom_pct  orders_mom_pct
      2023-01      66                59     143,944.00   131,985.70       11,958.30         286         1,999.78           8.31          NaN             NaN
      2023-02      61                52     115,420.00   105,521.80        9,898.20         220         1,729.87           8.58       -20.10           -7.60
      2023-03      68                63     143,777.00   132,792.05       10,984.95         283         1,952.82           7.64        25.80           11.50
      2023-04      56                52     115,440.00   104,787.10       10,652.90         210         1,871.20           9.23       -21.10          -17.60
      2023-05      61                57     108,940.00    99,243.50        9,696.50         210         1,626.94           8.90        -5.30            8.90
      2023-06   

In [34]:
# ── 6B: Product × Category × Month Pivot (for BI tools) ──────────────────────
dash_product = active.groupby(['order_yearmon', 'category', 'product'], observed=True).agg(
    orders    = ('order_id', 'count'),
    net_rev   = ('net_revenue', 'sum'),
    units     = ('quantity', 'sum')
).round(2).reset_index()

print('Product Performance Table shape:', dash_product.shape)
print(dash_product.head(8).to_string(index=False))

Product Performance Table shape: (288, 6)
order_yearmon  category         product  orders   net_rev  units
      2023-01 Computing    Laptop Basic       4 11,673.30     18
      2023-01 Computing      Laptop Pro       7 35,332.80     30
      2023-01 Computing     Monitor 24"       5  6,980.00     21
      2023-01 Computing     Monitor 27"       7 17,293.50     34
      2023-01 Computing  Tablet Premium       7 29,802.70     41
      2023-01 Computing Tablet Standard       6 10,872.75     29
      2023-01    Mobile Smartphone Lite       2  3,277.70      8
      2023-01    Mobile    Smartphone X       3  7,956.15     10


In [35]:
# ── 6C: Customer 360 Table (one row per customer) ─────────────────────────────
customer_360 = active.groupby('customer_id').agg(
    total_orders    = ('order_id', 'count'),
    total_revenue   = ('net_revenue', 'sum'),
    avg_order_value = ('net_revenue', 'mean'),
    first_order     = ('order_date', 'min'),
    last_order      = ('order_date', 'max'),
    favourite_product = ('product', lambda x: x.mode()[0]),
    countries_ordered = ('ship_country', 'nunique'),
    avg_discount    = ('discount_pct', 'mean')
).round(2).reset_index()

customer_360['customer_lifetime_days'] = (
    customer_360['last_order'] - customer_360['first_order']
).dt.days

customer_360['recency_days'] = (
    snapshot_date - customer_360['last_order']
).dt.days

# Merge RFM segment
customer_360 = customer_360.merge(
    rfm[['segment']].reset_index(),
    on='customer_id', how='left'
)

print('Customer 360 Table shape:', customer_360.shape)
print(customer_360.head(5).to_string(index=False))

Customer 360 Table shape: (298, 12)
 customer_id  total_orders  total_revenue  avg_order_value first_order last_order   favourite_product  countries_ordered  avg_discount  customer_lifetime_days  recency_days             segment
        1001             5       3,189.65           637.93  2023-02-14 2024-09-15 Mechanical Keyboard                  3          8.00                     579           107 Potential Loyalists
        1002             8      20,112.05         2,514.01  2023-05-07 2024-06-29        Laptop Basic                  5         11.88                     419           185     Loyal Customers
        1003             6       9,764.00         1,627.33  2023-01-17 2024-09-07          Laptop Pro                  4          9.17                     599           115 Potential Loyalists
        1004             7      12,459.60         1,779.94  2023-06-03 2024-10-24      Tablet Premium                  5          6.43                     509            68     Loyal Customers

In [40]:
! uv pip install -q pyarrow fastparquet

In [37]:
# ── Save all dashboard tables ─────────────────────────────────────────────────
import os

output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

# Save to Parquet (for BI tools like Power BI / Tableau)
clean.to_parquet(f'{output_dir}/techmart_gold_layer.parquet', index=False)
dash_monthly.to_parquet(f'{output_dir}/dashboard_monthly.parquet', index=False)
dash_product.to_parquet(f'{output_dir}/dashboard_product.parquet', index=False)
customer_360.to_parquet(f'{output_dir}/customer_360.parquet', index=False)
rfm.reset_index().to_parquet(f'{output_dir}/rfm_segments.parquet', index=False)

# Also save to CSV for sharing with non-technical stakeholders
dash_monthly.to_csv(f'{output_dir}/dashboard_monthly.csv', index=False)
customer_360.to_csv(f'{output_dir}/customer_360.csv', index=False)
kpi_df.to_csv(f'{output_dir}/kpi_summary.csv')

print('All output files saved to /output/')
for f in os.listdir(output_dir):
    size = os.path.getsize(f'{output_dir}/{f}')
    print(f'  {f:<40} {size/1024:.1f} KB')

All output files saved to /output/
  customer_360.parquet                     20.3 KB
  dashboard_monthly.csv                    1.7 KB
  customer_360.csv                         25.3 KB
  rfm_segments.parquet                     12.1 KB
  dashboard_product.parquet                7.1 KB
  kpi_summary.csv                          0.4 KB
  dashboard_monthly.parquet                8.6 KB
  techmart_gold_layer.parquet              60.3 KB


### STEP 7 — Executive Summary
*This is the narrative you would write in your stakeholder report.*

In [38]:
total_net = active['net_revenue'].sum()
top_product = product_perf['net_rev'].idxmax()
top_country = country_perf['net_rev'].idxmax()
champions_rev = seg_summary.loc['Champions', 'total_revenue'] if 'Champions' in seg_summary.index else 0
champions_count = seg_summary.loc['Champions', 'customer_count'] if 'Champions' in seg_summary.index else 0

print(f"""
╔══════════════════════════════════════════════════════════════╗
║          TECHMART ANALYTICS — EXECUTIVE SUMMARY             ║
╚══════════════════════════════════════════════════════════════╝

BUSINESS PERFORMANCE
  • Total Net Revenue    : ${total_net:>12,.0f}
  • Active Orders        : {len(active):>12,}
  • Unique Customers     : {active['customer_id'].nunique():>12,}
  • Avg Order Value      : ${active['net_revenue'].mean():>12,.0f}

TOP PERFORMERS
  • Best Product         : {top_product}
  • Best Country         : {top_country}
  • Champion Customers   : {int(champions_count)} customers generating 
                           ${champions_rev:,.0f} revenue

DATA QUALITY ACTIONS TAKEN
  • Removed 15 duplicate orders
  • Fixed 120 missing values across 3 columns
  • Standardized country names (12 → 6 unique)
  • Removed invalid quantities (negative/zero)
  • Capped price outliers via product-median imputation

RECOMMENDED ACTIONS
  1. Re-engage 'At Risk' and 'Lost' customer segments
  2. Increase inventory for top product line
  3. Investigate cancellation rates in high-discount orders
  4. Expand presence in {top_country} (highest revenue market)
""")


╔══════════════════════════════════════════════════════════════╗
║          TECHMART ANALYTICS — EXECUTIVE SUMMARY             ║
╚══════════════════════════════════════════════════════════════╝

BUSINESS PERFORMANCE
  • Total Net Revenue    : $   2,783,742
  • Active Orders        :        1,578
  • Unique Customers     :          298
  • Avg Order Value      : $       1,764

TOP PERFORMERS
  • Best Product         : Laptop Pro
  • Best Country         : United States
  • Champion Customers   : 53 customers generating 
                           $831,242 revenue

DATA QUALITY ACTIONS TAKEN
  • Removed 15 duplicate orders
  • Fixed 120 missing values across 3 columns
  • Standardized country names (12 → 6 unique)
  • Removed invalid quantities (negative/zero)
  • Capped price outliers via product-median imputation

RECOMMENDED ACTIONS
  1. Re-engage 'At Risk' and 'Lost' customer segments
  2. Increase inventory for top product line
  3. Investigate cancellation rates in high-discount o